# Customer Churn Prediction - Google Colab Workspace

Use this notebook to run, test, edit, and push this project from Google Colab.

**Repository:** https://github.com/mudather490/Customer-Churn-Prediction-as-Project-02-

| Step | What it does |
|---|---|
| 1 | Clone or update the repository |
| 2 | Install the pinned dependencies |
| 3 | Verify the trained model artifact |
| 4 | Run the 92-test suite |
| 5 | Try predictions in-process |
| 6 | Optionally serve Swagger UI on a Colab URL |
| 7 | Push your edits back to GitHub |

> The trained model artifact is included in the repository, so no extra setup is required.


## 1. Clone or update the repository


In [ ]:
REPO_URL = "https://github.com/mudather490/Customer-Churn-Prediction-as-Project-02-.git"
PROJECT_DIR = "/content/Customer-Churn-Prediction-as-Project-02-"

import os
import subprocess
from pathlib import Path

if Path(PROJECT_DIR, ".git").is_dir():
    result = subprocess.run(
        ["git", "-C", PROJECT_DIR, "pull", "--ff-only"],
        capture_output=True,
        text=True,
    )
else:
    result = subprocess.run(
        ["git", "clone", REPO_URL, PROJECT_DIR],
        capture_output=True,
        text=True,
    )

print(result.stdout or result.stderr)
os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())
print(
    subprocess.run(
        ["git", "log", "--oneline", "-3"], capture_output=True, text=True
    ).stdout
)


## 2. Install dependencies


In [ ]:
%pip install -q -r requirements-dev.txt


> **If you see a NumPy, pandas, or scikit-learn binary error**, restart the runtime
> (`Runtime -> Restart session`) and run this cell again. You do not need to re-clone
> unless the files are gone.


## 3. Verify the model artifact


In [ ]:
import hashlib

import joblib
from pathlib import Path

EXPECTED_SHA256 = "C4D73339D823F3FBA77FECC7235DF7DAC31E9044D8B80B5332949C0D459D503E"

artifact = Path("models/customer_churn_model.joblib")
assert artifact.is_file(), "Model artifact missing. Run the clone cell first."

actual = hashlib.sha256(artifact.read_bytes()).hexdigest().upper()
print("Path:    ", artifact.resolve())
print("Size:    ", artifact.stat().st_size, "bytes")
print("SHA256:  ", actual)
assert actual == EXPECTED_SHA256, "Artifact hash does not match the trusted copy."

bundle = joblib.load(artifact)
print("Model:   ", bundle["model_name"])
print("Threshold:", bundle["threshold"])
print("Features:", len(bundle["feature_columns"]))


## 4. Run the test suite


In [ ]:
!python -m pytest -q


## 5. Try predictions in-process


In [ ]:
from fastapi.testclient import TestClient

from app.main import app

low_risk_customer = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "Yes",
    "tenure": 60,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "DSL",
    "OnlineSecurity": "Yes",
    "OnlineBackup": "Yes",
    "DeviceProtection": "Yes",
    "TechSupport": "Yes",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Two year",
    "PaperlessBilling": "No",
    "PaymentMethod": "Bank transfer (automatic)",
    "MonthlyCharges": 55.0,
    "TotalCharges": 3300.0,
}

high_risk_customer = {
    "gender": "Male",
    "SeniorCitizen": 0,
    "Partner": "No",
    "Dependents": "No",
    "tenure": 1,
    "PhoneService": "Yes",
    "MultipleLines": "Yes",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 95.0,
    "TotalCharges": 95.0,
}

with TestClient(app) as client:
    print("HEALTH:", client.get("/health").json())
    print("MODEL :", client.get("/model-info").json()["model_name"])
    print("LOW   :", client.post("/predict", json=low_risk_customer).json())
    print("HIGH  :", client.post("/predict", json=high_risk_customer).json())


## 6. Serve the API on a Colab URL (optional)

This runs uvicorn in a background thread and opens a temporary Colab proxy link.
Add `/docs` to the generated URL to open Swagger UI.


In [ ]:
import threading
import time

import uvicorn
from google.colab.output import serve_kernel_port_as_window

from app.main import app


def _serve():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")


threading.Thread(target=_serve, daemon=True).start()
time.sleep(3)

print("Open the link below, then add /docs to the URL for Swagger UI.")
serve_kernel_port_as_window(8000, anchor_text="Open the API root")


## 7. Push your edits back to GitHub

### Edit files

1. Open the file browser on the left side of Colab.
2. Navigate to `/content/Customer-Churn-Prediction-as-Project-02-`.
3. Double-click any `.py`, `.md`, or `.ini` file to edit it.
4. Re-run the tests before pushing.

### Create the token

1. Open https://github.com/settings/tokens and create a token with `repo` scope.
2. In Colab, open the key icon (**Secrets**) in the left sidebar.
3. Add a secret named `GITHUB_TOKEN` and paste the token.
4. Enable notebook access for that secret.

Then run the cell below. It commits every change and pushes to `main`.


In [ ]:
import subprocess

from google.colab import userdata

TOKEN = userdata.get("GITHUB_TOKEN")
REMOTE = "https://github.com/mudather490/Customer-Churn-Prediction-as-Project-02-.git"
AUTH_REMOTE = (
    "https://x-access-token:"
    + TOKEN
    + "@github.com/mudather490/Customer-Churn-Prediction-as-Project-02-.git"
)


def run(args):
    result = subprocess.run(args, capture_output=True, text=True)
    output = (result.stdout + result.stderr).replace(TOKEN, "***")
    print(output.strip())
    return result.returncode


run(["git", "config", "user.name", "mudather490"])
run(["git", "config", "user.email", "mudatherkbyer@gmail.com"])
run(["git", "remote", "set-url", "origin", AUTH_REMOTE])
run(["git", "add", "-A"])
run(["git", "status", "--short"])

MESSAGE = "Update from Google Colab"
run(["git", "commit", "-m", MESSAGE])
push_code = run(["git", "push", "origin", "HEAD:main"])

# Remove the token from the stored remote URL.
run(["git", "remote", "set-url", "origin", REMOTE])

print()
print("Push succeeded." if push_code == 0 else "Push failed or nothing to commit. Read the output above.")


## Troubleshooting

| Problem | Fix |
|---|---|
| `ModuleNotFoundError` | Re-run the dependency cell, then restart the runtime if packages changed. |
| NumPy/pandas binary error | `Runtime -> Restart session`, then re-run from the install cell. |
| `FileNotFoundError` for the model | Re-run the clone cell; the artifact is tracked in the repository. |
| `pytest` command not found | Use `!python -m pytest -q` exactly as shown. |
| Push rejected | Run the clone/pull cell, resolve conflicts, then push again. |
| Colab URL expired | Re-run the serve cell; links are tied to the live session. |

Colab sessions are temporary. Push or download your work before closing the tab.
